# Method contrast — PTO vs GRPO at matched look-ahead  `[EVAL]`

**RQ-ii: does iterative GRPO compete with PTO under a matched look-ahead depth?** The method contrast
as persona-paired tables at each K, on **both graders side by side** (column `judge`: `gpt-4o-mini` =
the training oracle, `claude-haiku-4-5` = the held-out judge). Everything is full-conversation eval,
paired by the 96 shared personas. §0b first puts **all four arms at their endpoint, both graders, each
vs its own base** in one table (`headline_grid`) — the level check the method contrast presupposes.
How far each contrast's support runs is **derived from the frame**, never asserted in prose: the
`iteration` / `iter_a` / `iter_b` columns name it, and §3b's `meta.censoring` key states it.
Exports → `results/method/contrast/{tables,figures}/` (judge-invariant family: no `<judge>/` level —
the grader is a column, never a folder). Ported from `7_Stats` §4a/§4b; the same rows for the same arm
and grader match the retired `results/L0|L5/tables/7_stats/<judge>/` tables.

**Sign convention:** `+ mean_delta ⇒ PTO higher` in §1–§3. ⚠ In **§0b the sign means something else** —
there `+ delta ⇒ the arm's endpoint beat its OWN base`, not that one method beat the other. On `MICI`
(MI-inconsistent behaviour, lower = better) a positive Δ is *worse* in both sections; on every other
rubric it is better.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting, stats
from eda_analysis.constants import judge_dirname, PRIMARY_JUDGE_TAG, LOWER_IS_BETTER
cfg = eda_analysis.EdaConfig(family="method/contrast", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the banner reset_results just removed (judge-invariant: lives directly under figures/)

## 0 · Confirmatory vs exploratory — read this first  `[EVAL]`

**Confirmatory (a thesis claim).** *PTO > GRPO on Q1+Q2 at matched budget* — reported at **both** the
matched iteration (§1, `method_paired_by_K`: every iteration both arms reached) **and** as the
**best-vs-best model-selection contrast** (§2, `method_paired_best`: PTO at its own-oracle best iteration vs
GRPO at *its* best), so GRPO is credited at its peak rather than only at its regressed endpoint. The
held-out grader is the out-of-sample check on the same claim: it never scored a training reward, so a
gap that survives it is not the oracle grading its own homework.

**§0b's status.** `headline_grid` is the **level context**, not a method verdict: the confirmatory claim
it carries is *each arm improves over its OWN base* on Q1+Q2 under both graders — a claim owned per
grader by `arms/stats` (`main_results`, `target=final`), which §0b restates with the two graders side by
side because that is the join every reader would otherwise do by hand. Its non-`Q1Q2` rows are
exploratory, exactly as they are there. Reading a PTO-vs-GRPO verdict out of §0b would be comparing two
arms' *anchored* levels rather than a paired contrast — §1/§2 are the contrast, and their sign depends
on K.

**Multiplicity scope.** Each row's `p_holm` is Holm-corrected across the rubrics **within its own
(judge, K, iteration) contrast** — every matched-budget point is its own family; corrections are **not**
pooled across iterations or graders (§0b corrects within its own (judge, arm) family). `mean_delta` /
`dz` are unaffected by the correction.

**Exploratory.** Every non-`Q1Q2` rubric here (WAI-SR, CSQ-8, MI-SAT, MITI, PCT, MICI, Q1, Q2) is
hypothesis-generating; the K=5 rows are matched on *iteration*, not *budget* — the budget-matched
form of the same contrast lives in `compute/cost` (§4 pointer).

In [ ]:
SC = eda_analysis.scores_by_judge(S)          # {judge_label: scores_long}, primary FIRST
PRIMARY = judge_dirname(PRIMARY_JUDGE_TAG)
JUDGES = list(SC)
ROLE = {j: ("training oracle" if j == PRIMARY else "held-out judge") for j in JUDGES}
for j, sc in SC.items():
    print(f"{j:18s} ({ROLE[j]:15s}) scores_long {sc.shape} | arms {sorted(sc.arm.unique())} | "
          f"K {sorted(sc.K.unique())}")
KS = sorted(SC[PRIMARY].K.unique())
print("K levels:", KS)

## 0b · Where the four arms land — the endpoint grid  `[EVAL]`
**Purpose.** The headline artifact: **all four arms, at their final iteration, on every rubric, under
BOTH graders, in ONE table**, each arm anchored to **its own base**. It is what a paper, a deck or a
supervisor email opens with, and nothing else in `results/` carries it — the per-judge leaderboards
(`results/arms/stats/tables/<judge>/main_results.md`) split the two graders across two files, and
hand-joining those is exactly where a reader is tempted to average the graders (which is forbidden:
they are train-vs-test, not two raters). This family is the right home because it is judge-invariant:
the grader is a **column**, not a folder. Sections 1–3 below then take the same endpoint apart as the
*method* contrast; this one is the vs-base level check that has to be true before any of that matters.

**Columns.** `judge` (`gpt-4o-mini` = the training oracle, `claude-haiku-4-5` = the held-out judge) ·
`arm` / `method` / `K` · `final_iter` (that arm's last scored iteration) · `metric` · `n` = paired
personas · `base_mean` and `final_mean` (that arm's OWN base and its endpoint, over the paired
personas) · `delta` = `final_mean − base_mean` · `dz` = paired Cohen's *d* · `ci_lo`/`ci_hi` =
persona-bootstrap 95 % CI on `delta` · `p` = Wilcoxon signed-rank · `p_holm` = Holm **across the
rubrics within each (judge, arm)** — the same family scope `arms/stats`'s `main_results` uses, so the
primary-grader rows of this table and that one are the same statistic and must agree.

**Traps this table is built around.**
- **Each arm is anchored to ITS OWN base**, not to a shared/pooled base. The four arms each generated
  their own 96 base conversations, so their base means genuinely differ (Q1+Q2 under the primary
  grader: `GRPO_LA5` 2.963 … `GRPO_LA0` 3.067). One pooled anchor would shift each arm's `delta` by a
  different amount and quietly re-rank them.
- **Paired on `persona_id`, never `file_index`.** The 96 personas are reshuffled every iteration, so a
  `file_index` join would pair unrelated conversations — means survive it, `dz` and CIs do not.
- **Levels are NOT comparable across the `judge` rows, and must never be averaged.** The held-out
  grader sits ≈1.0–1.7 points lower on Q1/Q2/MITI and slightly *higher* on MICI, and the offset is
  model-dependent (`results/measurement/validity/tables/second_judge_agreement.md`, column
  `bias_judge_minus_primary`). Compare `delta` / `dz` across graders; compare `base_mean` /
  `final_mean` only *within* a grader.
- **`MICI` is lower-is-better.** No sign is flipped here (same convention as `main_results`), so a
  positive `delta` on MICI means the arm got **worse**.
- **This is a vs-base table, not a method verdict.** PTO-vs-GRPO is §1/§2, and its sign flips with K —
  never quote a row of this grid as "PTO beats GRPO".
- `ci_lo`/`ci_hi` come from `stats.paired_arrays`' seeded percentile bootstrap, whereas `main_results`
  uses `stats.bootstrap_ci`'s (also `BOOT_SEED`-seeded) resampler; both are reproducible but they are
  different draws, so the CI *bounds* differ from that table by up to ~0.02 score points while
  `base_mean`, `delta` and `dz` match exactly.

In [ ]:
ARM_ORDER = ["PTO_LA0", "PTO_LA5", "GRPO_LA0", "GRPO_LA5"]   # display order: PTO first, K ascending


def persona_pivot(sc, arm, metric):
    """`persona_id` x `iteration` table for one arm on one rubric — THE pairing unit.

    persona_id, never file_index: the 96 personas are reshuffled every iteration.
    """
    a = sc[(sc.arm == arm) & (sc.questionnaire == metric)]
    return a.pivot_table(index="persona_id", columns="iteration", values="score")


def arm_endpoints(sc, arm):
    """(base_iteration, final_iteration) for one arm — its OWN base, its last scored state."""
    a = sc[sc.arm == arm]
    if a.empty or not a.is_base.any() or not (~a.is_base).any():
        return None, None
    return int(a.loc[a.is_base, "iteration"].iloc[0]), int(a.loc[~a.is_base, "iteration"].max())


def endpoint_grid(SC, judges, arm_order=ARM_ORDER):
    """One row per (judge, arm, metric): the arm's endpoint vs ITS OWN base, persona-paired."""
    rows = []
    for j in judges:
        sc = SC[j]
        arms = ([a for a in arm_order if a in set(sc.arm)] +
                [a for a in sorted(sc.arm.unique()) if a not in arm_order])
        for arm in arms:
            base_it, fin_it = arm_endpoints(sc, arm)
            if base_it is None:
                continue                      # an arm with no base (or no trained state) can't anchor
            a = sc[sc.arm == arm]
            arm_rows = []
            for m in [x for x in eda_analysis.QUESTIONNAIRE_ORDER if x in set(a.questionnaire)]:
                piv = persona_pivot(sc, arm, m)
                if base_it not in piv.columns or fin_it not in piv.columns:
                    continue
                b, f = piv[base_it].to_numpy(float), piv[fin_it].to_numpy(float)
                ok = ~(np.isnan(b) | np.isnan(f))          # the same pairwise drop paired_arrays does
                st = stats.paired_arrays(f, b)             # + => the endpoint scores HIGHER than base
                arm_rows.append({
                    "judge": j, "arm": arm, "method": str(a.method.iloc[0]), "K": int(a.K.iloc[0]),
                    "final_iter": fin_it, "metric": m, "n": st["n"],
                    "base_mean": float(np.mean(b[ok])), "final_mean": float(np.mean(f[ok])),
                    "delta": st["mean_delta"], "dz": st["dz"],
                    "ci_lo": st["ci_lo"], "ci_hi": st["ci_hi"], "p": st["p"]})
            if arm_rows:                       # Holm across the RUBRICS within this (judge, arm)
                for r, ph in zip(arm_rows, stats.holm(np.array([r["p"] for r in arm_rows], float))):
                    r["p_holm"] = float(ph)
                rows += arm_rows
    if not rows:
        return pd.DataFrame()
    HG = pd.DataFrame(rows)
    for col, cats in (("judge", judges), ("arm", arm_order),
                      ("metric", eda_analysis.QUESTIONNAIRE_ORDER)):
        present = [c for c in cats if c in set(HG[col])] + \
                  [c for c in dict.fromkeys(HG[col]) if c not in cats]
        HG[col] = pd.Categorical(HG[col], categories=present, ordered=True)
    HG = HG.sort_values(["judge", "arm", "metric"], kind="stable").reset_index(drop=True)
    for col in ("judge", "arm", "metric"):
        HG[col] = HG[col].astype(str)
    return HG


HG = endpoint_grid(SC, JUDGES)
if HG.empty:
    print("no arm has both an own base and a trained state under any grader.")
else:
    HGv = HG[["judge", "arm", "method", "K", "final_iter", "metric", "n", "base_mean",
              "final_mean", "delta", "dz", "ci_lo", "ci_hi", "p", "p_holm"]].round(4)
    print(f"=== endpoint vs own base: {HG.judge.nunique()} graders x {HG.arm.nunique()} arms x "
          f"{HG.metric.nunique()} metrics = {len(HG)} rows (+ => endpoint higher) ===")
    display(HGv[HGv.metric == "Q1Q2"])
    exports.save_table(HGv, "headline_grid", caption=(
        "THE endpoint grid: all four arms at their final iteration on every rubric, under BOTH graders "
        "in one judge-invariant table (column `judge`: gpt-4o-mini = the training oracle, "
        "claude-haiku-4-5 = the held-out judge). Each arm is anchored to ITS OWN base (iteration 0 of "
        "that arm, its own 96 base conversations) - not to a shared or pooled base - and every contrast "
        "is paired on persona_id over the 96 shared personas (never file_index: the personas reshuffle "
        "each iteration). delta = final_mean - base_mean, dz = paired Cohen's d, ci_lo/ci_hi = "
        "persona-bootstrap 95% CI on delta (BOOT_SEED), p = Wilcoxon signed-rank, p_holm = Holm across "
        "the rubrics within each (judge, arm). + => the endpoint scores higher; MICI is lower-better so "
        "+ there means WORSE (no sign is flipped). WARNING: the two graders are NOT on a common scale "
        "and must never be averaged - the held-out judge's level offset is model-dependent - so compare "
        "delta/dz across graders and levels only within a grader. The primary-grader rows are the same "
        "statistic as results/arms/stats/tables/gpt-4o-mini/main_results.md (target=final) and agree "
        "with it on base_mean, delta, dz and p_holm."))

**The same endpoint as a picture (`headline_grid`).** The Q1+Q2 *level* of all four arms at their
final iteration, **one panel per grader**, primary (the training oracle) on the left. Filled marker =
the endpoint with a persona-bootstrap 95 % CI; hollow grey marker = **that arm's own base**, with its
own CI; the stem between them is the gain the table calls `delta`. ⚠ **The panels are on INDEPENDENT
y-axes and are not a common scale** — the held-out grader's level offset is large and
model-dependent, so a bar that looks taller on the right panel says nothing about the left one. Read
*within* a panel (which arm is higher, how far above its own base) and read the cross-grader question
off `delta`/`dz` in the table. CIs are computed explicitly with `stats.bootstrap_ci` (seeded with
`constants.BOOT_SEED`) and drawn as error bars — no seaborn CI estimator is involved, so the figure is
byte-reproducible across renders.

In [ ]:
def headline_grid_fig(SC, judges, metric="Q1Q2", arm_order=ARM_ORDER):
    # Endpoint LEVEL per arm, one panel per grader. INDEPENDENT y-axes on purpose (see the note
    # above): the graders' levels are offset by a model-dependent amount and must not share a scale.
    arms = [a for a in arm_order if a in set(SC[judges[0]].arm)]
    if not arms:
        return None
    pal = plotting.arm_palette(arms)
    fig, axes = plt.subplots(1, len(judges), figsize=(4.0 * len(judges), 3.8), squeeze=False)
    for ji, jn in enumerate(judges):
        ax, sc = axes[0, ji], SC[jn]
        ticks = []
        for xi, arm in enumerate(arms):
            base_it, fin_it = arm_endpoints(sc, arm)
            piv = persona_pivot(sc, arm, metric)
            if base_it is None or base_it not in piv.columns or fin_it not in piv.columns:
                ticks.append(eda_analysis.arm_label(arm))
                continue
            b = piv[base_it].dropna().to_numpy(float)
            f = piv[fin_it].dropna().to_numpy(float)
            blo, bhi = stats.bootstrap_ci(b)          # persona bootstrap, seeded with BOOT_SEED
            flo, fhi = stats.bootstrap_ci(f)
            bm, fm = float(b.mean()), float(f.mean())
            col = pal.get(arm, "0.4")
            ax.plot([xi, xi], [bm, fm], color=col, lw=2.0, alpha=0.45, zorder=1, solid_capstyle="butt")
            ax.errorbar(xi, bm, yerr=[[bm - blo], [bhi - bm]], fmt="o", ms=6.5, mfc="white",
                        mec="0.45", ecolor="0.55", elinewidth=1.1, capsize=3, zorder=2)
            ax.errorbar(xi, fm, yerr=[[fm - flo], [fhi - fm]], fmt="o", ms=8.5, color=col,
                        ecolor=col, elinewidth=1.9, capsize=3.5, zorder=3)
            ax.annotate(f"{fm - bm:+.2f}", (xi, fm), textcoords="offset points", xytext=(9, 2),
                        fontsize=8, color=col, fontweight="bold")
            ticks.append(f"{eda_analysis.arm_label(arm)}\nI{fin_it}")
        ax.set_xticks(range(len(arms)))
        ax.set_xticklabels(ticks, fontsize=8)
        ax.set_xlim(-0.6, len(arms) - 0.2)
        ax.set_title(f"{jn} ({ROLE.get(jn, '')})", fontsize=9.5)
        ax.set_ylabel(f"{eda_analysis.display_label(metric)} — mean score", fontsize=9)
        ax.tick_params(labelsize=8)
        ax.grid(True, axis="y", alpha=0.35)
    h = [Line2D([], [], color="0.25", marker="o", ms=8, ls="none",
                label="endpoint (that arm's final iteration), 95% CI"),
         Line2D([], [], color="0.45", marker="o", ms=6.5, ls="none", mfc="white",
                label="that arm's OWN base (iteration 0), 95% CI"),
         Line2D([], [], color="0.45", lw=2.0, alpha=0.6, label="the gain (= delta in headline_grid)")]
    lg = fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False,
                    fontsize=8, title_fontsize=7.5,
                    title=(f"{eda_analysis.short_label(metric)} at the endpoint, one panel per grader. "
                           "CIs = persona bootstrap, n = 96 personas.   "
                           "WARNING: INDEPENDENT y-axes - the panels are NOT on a common scale."))
    lg.get_title().set_color("0.3")
    fig.tight_layout()
    return fig


if not HG.empty:
    fig = headline_grid_fig(SC, JUDGES, metric="Q1Q2")
    if fig is not None:
        exports.save_fig(fig, "headline_grid", caption=(
            "Where the four arms land on Q1+Q2 at their final iteration, one panel per grader (left "
            "gpt-4o-mini = the training oracle, right claude-haiku-4-5 = the held-out judge). Filled "
            "marker = the endpoint, hollow grey marker = THAT ARM'S OWN base (iteration 0 of the same "
            "arm - not a shared base), stem = the gain; error bars are persona-bootstrap 95% CIs over "
            "the 96 shared personas (BOOT_SEED), paired on persona_id. The number beside each endpoint "
            "is that arm's delta over its own base. WARNING: the two panels have INDEPENDENT y-axes and "
            "are NOT on a common scale - the held-out judge's level offset is large and "
            "model-dependent, so heights are comparable only WITHIN a panel and the graders' raw scores "
            "must never be averaged. The numbers are the Q1Q2 rows of tables/headline_grid.md."))
        plt.show()

**Ledger keys (`headline.*`).** The Q1+Q2 cell of every (grader × arm) as citable keys, written into
the **same** ledger `results/method/contrast/tables/method_contrast.json` that §3b fills — `save_numbers`
merges by key, so `headline.*` sits beside `matched_last.*` / `best_vs_best.*` rather than replacing
them. Every value is a cell of `tables/headline_grid.md`; nothing is computed here that is not in that
table. Key shape: `headline.<judge>.<arm>.Q1Q2.<field>`.

In [ ]:
if not HG.empty:
    HNUM, src = {}, "tables/headline_grid.md"
    q = HG[HG.metric == "Q1Q2"]
    for _, r in q.iterrows():
        pre = f"headline.{r.judge}.{r.arm}.Q1Q2"
        note = (f"{r.arm} at iteration {int(r.final_iter)} vs ITS OWN base, graded by {r.judge}; "
                f"persona-paired n = {int(r.n)}; + => the endpoint scores higher")
        for field, val in (("final_iter", int(r.final_iter)), ("n", int(r.n)),
                           ("base_mean", float(r.base_mean)), ("final_mean", float(r.final_mean)),
                           ("delta", float(r.delta)), ("dz", float(r.dz)),
                           ("ci_lo", float(r.ci_lo)), ("ci_hi", float(r.ci_hi)),
                           ("p_holm", float(r.p_holm))):
            HNUM[f"{pre}.{field}"] = {"value": val, "source": src, "note": note}
    HNUM["headline.meta.grid"] = {
        "value": (f"{HG.judge.nunique()} graders x {HG.arm.nunique()} arms x {HG.metric.nunique()} "
                  f"metrics = {len(HG)} rows"), "source": src, "note": "shape of headline_grid"}
    HNUM["headline.meta.anchor"] = {
        "value": "each arm vs ITS OWN base (iteration 0 of that arm), never a shared/pooled base",
        "source": src, "note": ""}
    HNUM["headline.meta.pairing"] = {
        "value": "paired on persona_id (never file_index - personas reshuffle each iteration), "
                 "n = 96 shared personas", "source": src, "note": ""}
    HNUM["headline.meta.scale"] = {
        "value": "levels are NOT comparable across graders and must never be averaged (train-vs-test, "
                 "model-dependent offset); compare delta/dz across graders, levels within one",
        "source": src, "note": ""}
    HNUM["headline.meta.holm_scope"] = {
        "value": "across the rubrics within each (judge, arm)", "source": src, "note": ""}
    HNUM["headline.meta.sign"] = {
        "value": "+ => the endpoint scores higher than the arm's own base (on MICI, lower = better, "
                 "so + => WORSE); no sign is flipped in the table", "source": src, "note": ""}
    hpath = exports.save_numbers("method_contrast", HNUM)
    print(f"{len(HNUM)} headline.* keys ->", hpath)
    display(pd.DataFrame([{"key": k, "value": v["value"]} for k, v in HNUM.items()
                          if not k.startswith("headline.meta.")]).set_index("key"))

## 1 · Matched iteration — PTO − GRPO at each K, every iteration both arms reached  `[EVAL]`
**Purpose.** The method contrast as a paired table at matched iterations, per grader (iteration-0 base
rows dropped from the table — the two arms' bases are independent draws of the same model, and are
kept only as the noise floor in the §3 figure; §0b keeps them as each arm's anchor). Columns: `judge`,
`K`, `iteration`, `metric`, `n`, `mean_delta`, `dz`, `p_holm`. Persona-paired Wilcoxon signed-rank +
Cohen's *dz* + Holm across rubrics within each (judge, K, iteration). Each K's rows run to the last
iteration **both** of that K's arms reached — read that endpoint off the `iteration` column rather
than assuming one.

In [ ]:
frames = []
for j, sc in SC.items():
    for K in sorted(sc.K.unique()):
        CMP = stats.paired_method_comparison(sc, "PTO", "GRPO", K=int(K))
        if not CMP.empty:
            frames.append(CMP.assign(judge=j))
MP = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if MP.empty:
    print("no common PTO/GRPO iterations at any K under any grader.")
else:
    MP["judge"] = pd.Categorical(MP["judge"], categories=JUDGES, ordered=True)
    MP = MP.sort_values(["judge", "K", "iteration"], kind="stable").reset_index(drop=True)
    MP["judge"] = MP["judge"].astype(str)
    MPt = MP[MP.iteration > 0]
    view = MPt[["judge", "K", "iteration", "metric", "n", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== PTO - GRPO at matched K + iterations, per grader (+ => PTO higher) ===")
    display(view[view.metric.isin(["Q1Q2", "MICI"])])
    exports.save_table(view, "method_paired_by_K", caption=(
        "PTO - GRPO at matched K and matched iterations, BOTH graders in one table (column `judge`: "
        "gpt-4o-mini = the training oracle, claude-haiku-4-5 = the held-out judge; column `K` = look-ahead "
        "depth). Persona-paired (n = 96 shared personas) Wilcoxon + Cohen's dz + Holm. + => PTO higher; on "
        "MICI (lower = better) a positive delta means PTO is WORSE. Holm scope: p_holm is corrected across "
        "the rubrics WITHIN each (judge, K, iteration) contrast, NOT across iterations or graders (each "
        "matched-budget point is its own family). Each K's rows run to the last iteration BOTH of that K's "
        "arms reached - the iteration column names it. Matched on ITERATION, not budget - see compute/cost "
        "for the budget-matched method sweeps. Same statistic as the retired "
        "results/L0|L5/tables/7_stats/<judge>/method_paired_by_K."))

## 2 · Best-vs-best — the model-selection contrast  `[EVAL]`
**Purpose.** PTO at its **own-oracle best** iteration vs GRPO at **its** best (per `best_per_experiment`,
selected under *that grader's* scores — so the held-out row selects checkpoints by the held-out grader),
persona-paired across the different iterations (valid: every iteration reshuffles the same 96 personas).
This is the strongest-steelman comparison: GRPO is credited at its peak, before the post-peak regression.
Complements the matched-iteration table above; `iter_a` = PTO's selected iteration, `iter_b` = GRPO's —
each side's best is drawn from the iterations that arm actually has, so read the selected iteration off
those two columns.

In [ ]:
frames = []
for j, sc in SC.items():
    for K in sorted(sc.K.unique()):
        BB = stats.paired_best_method_comparison(sc, "PTO", "GRPO", K=int(K))
        if not BB.empty:
            frames.append(BB.assign(judge=j))
BBall = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if BBall.empty:
    print("no K with both PTO and GRPO best models scored under any grader.")
else:
    BBall["judge"] = pd.Categorical(BBall["judge"], categories=JUDGES, ordered=True)
    BBall = BBall.sort_values(["judge", "K"], kind="stable").reset_index(drop=True)
    BBall["judge"] = BBall["judge"].astype(str)
    view_bb = BBall[["judge", "K", "iter_a", "iter_b", "metric", "n", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== PTO(best) - GRPO(best) per K and grader (+ => PTO higher; iter_a = PTO best, iter_b = GRPO best) ===")
    display(view_bb)
    exports.save_table(view_bb, "method_paired_best", caption=(
        "PTO at its own-oracle BEST iteration vs GRPO at ITS best, per K, BOTH graders in one table (column "
        "`judge`: gpt-4o-mini = the training oracle, claude-haiku-4-5 = the held-out judge; each grader's "
        "row selects the checkpoints under ITS OWN scores). Persona-paired (n = 96) Wilcoxon + dz + Holm "
        "across rubrics within each (judge, K). + => PTO higher; on MICI (lower = better) a positive delta "
        "means PTO is WORSE. The model-selection contrast: GRPO credited at its peak (iter_b), before the "
        "post-peak regression; complements method_paired_by_K. Each side's best is chosen from that arm's "
        "own scored support - iter_a / iter_b name the selected iterations. Same statistic as the retired "
        "results/L0|L5/tables/7_stats/<judge>/method_paired_best."))

## 3 · The method gap by iteration — one panel per grader  `[EVAL]`
**Purpose.** §1's `Q1Q2` rows as a picture: PTO − GRPO per iteration at K=0 (black, solid, circles) and
K=5 (green, dashed, squares), one panel per grader, ribbons = persona-bootstrap 95% CI, stars = cleared
Holm (across rubrics within that (judge, K, iteration) — the same `p_holm` as the table). Iteration 0 is
the two arms' independent base draws (hollow marker) and is the noise floor. Each K's line runs to that
K's last matched iteration, and the legend says so only when the two K's supports actually differ.
Rendered inline (the style of `plotting.k_did`'s method-gap row) from the §1 frame, so the figure never
disagrees with the table it sits next to.

In [ ]:
def method_gap_fig(MP, metric="Q1Q2", judges=None, gap_colors=None):
    # One column per grader; the K=0 / K=5 gap PTO - GRPO with CI ribbons + Holm stars.
    gap_colors = gap_colors or {0: "#111111", 5: "#009E73"}
    K_STYLE = plotting.lookahead.K_STYLE
    judges = judges or list(dict.fromkeys(MP["judge"]))
    sub_all = MP[MP.metric == metric]
    if sub_all.empty:
        return None
    n_it = int(sub_all.iteration.max()) + 1
    fig, axes = plt.subplots(1, len(judges), figsize=(3.6 * len(judges), 3.3), sharey=True, squeeze=False)
    for j, jn in enumerate(judges):
        ax = axes[0, j]
        for K in sorted(sub_all.K.unique()):
            sub = sub_all[(sub_all.judge == jn) & (sub_all.K == K)].sort_values("iteration")
            if sub.empty:
                continue
            st = K_STYLE.get(int(K), {"ls": "-", "marker": "o"})
            col = gap_colors.get(int(K), "0.4")
            ax.fill_between(sub.iteration, sub.ci_low, sub.ci_high, color=col, alpha=0.13, lw=0)
            pos = sub[sub.iteration > 0]
            ax.plot(pos.iteration, pos.mean_delta, ls=st["ls"], marker=st["marker"], color=col, lw=1.7, ms=5.5)
            base = sub[sub.iteration == 0]
            if not base.empty:            # the two independent base draws: hollow marker + a dotted link
                ax.plot(sub.iteration.iloc[:2], sub.mean_delta.iloc[:2], ls=":", color=col, lw=1.0)
                ax.scatter(base.iteration, base.mean_delta, marker=st["marker"], s=34, facecolor="white",
                           edgecolor=col, linewidth=1.3, zorder=4)
            sig = pos[pos.p_holm < 0.05]
            ax.scatter(sig.iteration, sig.mean_delta, marker="*", s=80, color=col, zorder=5,
                       edgecolor="white", linewidth=0.6)
        ax.axhline(0, color="0.35", lw=0.8)
        ax.set_title(f"{jn} ({ROLE.get(jn, '')})", fontsize=9.5)
        ax.set_xlabel("iteration (0 = the two arms' base draws)", fontsize=9)
        ax.set_xticks(range(0, n_it))
        ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.35)
    axes[0, 0].set_ylabel(f"{metric} Δ (PTO − GRPO), score points", fontsize=9)
    # Say "stops earlier" only when it IS earlier — DERIVE it, never hardcode which K ends first.
    k_end = {int(K): int(sub_all.loc[sub_all.K == K, "iteration"].max()) for K in sub_all.K.unique()}
    k5_note = (f" (to iter {k_end[5]}; the K=5 arms stop first)"
               if 5 in k_end and k_end[5] < max(k_end.values()) else "")
    h = [Line2D([], [], color=gap_colors[0], ls="-", marker="o", ms=5.5, lw=1.7, label="K=0: PTO_LA0 − GRPO_LA0"),
         Line2D([], [], color=gap_colors[5], ls="--", marker="s", ms=5.5, lw=1.7,
                label="K=5: PTO_LA5 − GRPO_LA5" + k5_note),
         Line2D([], [], color="0.2", ls="none", marker="*", ms=9, mec="white", mew=0.5,
                label="Holm p < .05 (across rubrics within the (judge, K, iteration) contrast)")]
    lg = fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False, fontsize=8,
                    title=f"{metric} gap PTO − GRPO by iteration, one panel per grader.  + => PTO higher.  "
                          "Ribbons = persona-bootstrap 95% CI, n = 96 personas.", title_fontsize=7.5)
    lg.get_title().set_color("0.3")
    fig.tight_layout()
    return fig

if not MP.empty:
    fig = method_gap_fig(MP, metric="Q1Q2", judges=JUDGES)
    if fig is not None:
        exports.save_fig(fig, "method_gap", caption=(
            "The method gap PTO - GRPO on Q1+Q2 per iteration, one panel per grader (left gpt-4o-mini = the "
            "training oracle, right claude-haiku-4-5 = the held-out judge): K=0 black solid circles, K=5 green "
            "dashed squares, ribbons = persona-bootstrap 95% CI over the 96 shared personas, stars = cleared "
            "Holm across rubrics within that (judge, K, iteration) contrast (the p_holm of method_paired_by_K). "
            "+ => PTO higher. Iteration 0 (hollow) is the two arms' independent base draws of the same model = "
            "the noise floor. Each K's line runs to that K's last matched iteration (the legend flags a shorter "
            "K=5 support only when there is one); matched on iteration, not budget. The endpoint LEVELS behind "
            "these differences are in headline_grid."))
        plt.show()

### 3b · Number ledger  `[EVAL]`
The headline cells of §1/§2 as citable keys (`results/method/contrast/tables/method_contrast.json`): for each
grader × K, the Q1+Q2 gap at the last matched iteration and at best-vs-best. Every value is a cell of the
two tables above (the `source` field says which); nothing is computed here that is not in a table.

In [ ]:
if not MP.empty:
    NUM = {}
    for j in JUDGES:
        for K in KS:
            m = MPt[(MPt.judge == j) & (MPt.K == K) & (MPt.metric == "Q1Q2")]
            if not m.empty:
                last = m[m.iteration == m.iteration.max()].iloc[0]
                pre = f"matched_last.{j}.K{int(K)}.Q1Q2"
                src = "tables/method_paired_by_K.md"
                note = f"PTO_LA{int(K)} - GRPO_LA{int(K)} at the last matched iteration ({int(last.iteration)}); + => PTO higher"
                NUM[f"{pre}.iteration"]  = {"value": int(last.iteration), "source": src, "note": note}
                NUM[f"{pre}.mean_delta"] = {"value": float(last.mean_delta), "source": src, "note": note}
                NUM[f"{pre}.dz"]         = {"value": float(last.dz), "source": src, "note": note}
                NUM[f"{pre}.p_holm"]     = {"value": float(last.p_holm), "source": src, "note": note}
                NUM[f"{pre}.n"]          = {"value": int(last.n), "source": src, "note": note}
            if not BBall.empty:
                b = BBall[(BBall.judge == j) & (BBall.K == K) & (BBall.metric == "Q1Q2")]
                if not b.empty:
                    b = b.iloc[0]
                    pre = f"best_vs_best.{j}.K{int(K)}.Q1Q2"
                    src = "tables/method_paired_best.md"
                    note = (f"PTO_LA{int(K)} at its best iteration ({int(b.iter_a)}) - GRPO_LA{int(K)} at its best "
                            f"({int(b.iter_b)}), each selected under this grader; + => PTO higher")
                    NUM[f"{pre}.iter_pto"]   = {"value": int(b.iter_a), "source": src, "note": note}
                    NUM[f"{pre}.iter_grpo"]  = {"value": int(b.iter_b), "source": src, "note": note}
                    NUM[f"{pre}.mean_delta"] = {"value": float(b.mean_delta), "source": src, "note": note}
                    NUM[f"{pre}.dz"]         = {"value": float(b.dz), "source": src, "note": note}
                    NUM[f"{pre}.p_holm"]     = {"value": float(b.p_holm), "source": src, "note": note}
                    NUM[f"{pre}.n"]          = {"value": int(b.n), "source": src, "note": note}
    NUM["meta.sign"] = {"value": "+ => PTO higher (on MICI, lower = better, + => PTO worse)", "source": "", "note": ""}
    NUM["meta.pairing"] = {"value": "persona-paired, n = 96 shared personas per contrast", "source": "", "note": ""}
    # DERIVED, never asserted: support_note() returns "" unless an arm really IS short relative to
    # the others in this frame, so this key can never claim a censoring the data does not have.
    NUM["meta.censoring"] = {
        "value": (eda_analysis.support_note(S.SCORES, label=False,
                                          subject="no later scored state")
                  or "no arm is short in this frame: every arm runs to the same last scored "
                     "iteration.") +
                 " Each contrast's own support is named by the iteration / iter_a / iter_b columns "
                 "of the tables above.",
        "source": "", "note": ""}
    NUM["meta.holm_scope"] = {"value": "across rubrics within each (judge, K, iteration) contrast", "source": "", "note": ""}
    path = exports.save_numbers("method_contrast", NUM, caption=(
        "Number ledger for the method contrast: the Q1+Q2 gap PTO - GRPO at the last matched iteration "
        "(matched_last.*) and at best-vs-best (best_vs_best.*), per grader x K, each key citing the table "
        "cell it was read from. + => PTO higher; persona-paired n = 96; how far each contrast's support "
        "actually ran is in the meta.censoring key, DERIVED from the frame rather than asserted here. The "
        "same file also carries the headline.* keys written by section 0b - the Q1+Q2 endpoint of each "
        "(grader, arm) vs THAT ARM'S OWN base, read off tables/headline_grid.md (there + => the endpoint "
        "beat its own base, not PTO beat GRPO)."))
    print(f"{len(NUM)} keys ->", path)
    display(pd.DataFrame([{"key": k, **v} for k, v in NUM.items() if not k.startswith("meta.")])
            .drop(columns=["note"]).set_index("key"))

## 4 · Where the budget-matched method contrast lives  `[EVAL]`
Both tables above match on **iteration**, which is not a fixed unit of spend: a whole PTO iteration costs a
fraction of a GRPO one (its dominant phase is the preference-tree *build*, and it needs no in-loop
reward computation), and a K=5 step costs ~1.9× a K=0 step. The same contrast at matched **GPU-hours** —
`method_K0` (PTO_LA0 vs GRPO_LA0) and `method_K5` (PTO_LA5 vs GRPO_LA5), each arm represented by the best
checkpoint reachable within a budget, on both graders and both selection metrics — is owned by
**`compute/cost`** (`compute.all_budget_sweeps` / `budget_sweep_crossjudge`; tables
`results/compute/cost/tables/budget_sweep_method_K{0,5}_*` and the `budget_sweep_crossjudge*` verdicts,
figure `budget_sweep_grid`). Quote the *sweep*, not a single iso-compute row: the sign of a lever is a
function of budget. This family deliberately does not recompute it — one owner per fact.

In [ ]:
_cost_tables = os.path.join(exports.RESULTS_DIR, "compute", "cost", "tables")
_hits = sorted(f for f in (os.listdir(_cost_tables) if os.path.isdir(_cost_tables) else [])
               if "method" in f and f.endswith(".md"))
if _hits:
    print("budget-matched method contrast tables rendered by compute/cost:")
    for f in _hits: print("   results/compute/cost/tables/" + f)
else:
    print("compute/cost has not rendered yet — run tools/render_results.py --family compute/cost "
          "for the budget-matched method sweeps.")

## 5 · Artifact index
Drop captions whose artifact no longer exists, then refresh `results/method/INDEX.md` + `results/INDEX.md`.

In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())